# Tutorial 09: Push Notification Pattern in A2A (Offline-First)

Focused tutorial for `A2A/09_Push-Notification`, centered on async task submission with optional polling fallback and push-style completion signaling.

## 1) Where We Are in the Journey

`08_streaming_a2a` emphasized continuous event delivery.
This step focuses on completion notification patterns for background tasks.
It exists to decouple execution lifecycle from request lifecycle.

## 2) What You Will Learn

- Submit/poll notification workflow
- Task ID lifecycle management
- Push callback vs polling fallback trade-off
- How this folder’s test flow models background completion

## 3) Why This Matters

Long-running tasks should not keep clients blocked.
Push-notification style patterns allow scalable background processing.
This improves throughput and UX for agentic workloads.

## 4) Core Concept Deep Dive

Folder test notebook submits via `/execute` and polls `/result/{task_id}`.
Architecturally, push systems still need durable task state and fallback reads.
Trade-off: push reduces polling overhead but needs callback reliability guarantees.

## 5) Code Walkthrough (Only `A2A/09_Push-Notification`)

- `test.ipynb` posts a query to controller endpoint.
- It extracts `task_id`, then polls status until `completed`.
- This tutorial mirrors that exact control flow with offline simulation.

In [ ]:
import time
import uuid

TASKS = {}
NOTIFICATIONS = []

def execute_async(query):
    task_id = str(uuid.uuid4())
    TASKS[task_id] = {'query': query, 'status': 'running', 'result': None, 'created': time.time()}
    return {'task_id': task_id, 'status': 'accepted'}

def maybe_complete(task_id):
    t = TASKS[task_id]
    if time.time() - t['created'] > 1.0 and t['status'] != 'completed':
        t['status'] = 'completed'
        t['result'] = 30 if 'add 10 and 20' in t['query'] else 'simulated-result'
        NOTIFICATIONS.append({'task_id': task_id, 'event': 'completed', 'result': t['result']})

def get_result(task_id):
    if task_id not in TASKS:
        return {'status': 'not_found'}
    maybe_complete(task_id)
    t = TASKS[task_id]
    return {'status': t['status'], 'result': t['result']}

In [ ]:
submit = execute_async('add 10 and 20')
print('SUBMIT:', submit)
task_id = submit['task_id']

while True:
    r = get_result(task_id)
    print('STATUS:', r)
    if r['status'] == 'completed':
        break
    time.sleep(0.4)

print('NOTIFICATIONS:', NOTIFICATIONS)

## 6) System Flow Explanation

1. Client submits task and receives task ID.
2. Worker executes asynchronously.
3. Task state moves toward terminal status.
4. Notification event is emitted on completion.
5. Client can still poll as fallback.

## 7) Important Concepts You Might Miss

- Push and polling are complementary, not mutually exclusive.
- Notification delivery should be idempotent.
- Task store durability is critical for correctness.

## 8) Common Mistakes / Pitfalls

- No fallback if callback endpoint is unavailable.
- Losing task state between submit and completion.
- Missing terminal statuses and error payloads.

## 9) Key Takeaways

- Push patterns decouple completion from request-response timing.
- Task IDs and durable state are foundational.
- Polling fallback remains essential for robustness.

## 10) What’s Next (Strict Preview)

`10_Human_in_loop` adds manual approval checkpoints inside the async workflow.
It solves when autonomous execution must pause for human decisions.
This matters for safety, compliance, and high-risk operations.